# QUEST alignment: how well does it point, and what do MAC and BLC add?

Every time the alignment is refitted (each sync), the driver logs its sync table: each point's observed direction
(the plate solve, Obs Az/Alt) and the raw motor pose it was taken at (p_az, p_alt, p_roll). This notebook replays
those tables (`utility/alignment_ab.py`) with the two alignment models and scores them by goto accuracy:

* **QUEST** — MAC off: the driver's weighted rotation-only QUEST (weights: proximity to the target, recency, polar).
* **Model** — MAC on: QUEST's rotation plus the M2/M3 zero offsets d2, d3 and the M2 scale s2
  (`driver/kinematics_mac.py`), fitted unweighted.

Questions:

1. **Which sessions' alignments are in the logs?**
2. **Once complete, how well does each point?** Each point held out and predicted from all the others.
3. **As the alignment grows, how well does the next goto land?** Each point predicted only from the points synced
   before it — what a goto to a new target sees during a session.
4. **Where does MAC help or hurt?** By altitude, roll and motor angles.
5. **One session in detail:** where its sync points were, and each one's error.
6. **What M2 scale and offsets do the alignments want?** And do d2, d3 stay put between sessions?
7. **What made a session point well: the offsets, MAC or Backlash Compensation (BLC)?**

The replay uses the logged raw motor pose, so neither model includes the worm feed-forward, the Sync Guide offset
(SGA) or the roll adjustment. BLC is different: it changes where the motors finish each goto, so it is already in the
sync points themselves (section 7 sorts the points by how BLC ended the goto before each). `docs/study_alignment_mac_v2.md`
has the study this replays.

In [ ]:
import os, sys, glob
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
sys.path.insert(0, os.path.abspath(os.path.join('.', '..', 'driver')))
import importlib, alignment_ab as ab
importlib.reload(ab)                                     # module changes without a kernel restart
from kinematics_mac import fit_alignment

PATTERN = '*.log'               # logs in LOG_DIR to replay, e.g. 'alpaca.jdm_Beta8.0_*.log'
LOG_DIR = ab.LOG_DIR            # logs/archive
S2 = ab.DEFAULT_M2_SCALE        # the model's M2 scale (fraction): the driver's align_m2_scale default
MIN_POINTS = 6                  # an alignment needs at least this many points
MIN_FIT = 3                     # section 3 predicts a point from at least this many earlier ones
MIN_SESSIONS = 5                # ... and plots a point count only when at least this many sessions reach it
SESSION = -1                    # the session section 5 shows first (index into the table in section 1)
TEMPLATE = 'plotly_dark'
COLOURS = {'quest': 'orange', 'model': 'deepskyblue'}
NAMES = {'quest': 'QUEST (MAC off)', 'model': 'Model (MAC on)'}
BLC_COLOURS = {'compensated': 'lightgreen', 'no slack': 'deepskyblue', 'given up': 'red', 'off': 'gray', 'no move': 'gray'}

def rms(x):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    return float(np.sqrt(np.mean(x ** 2))) if len(x) else np.nan

paths = sorted(glob.glob(os.path.join(LOG_DIR, PATTERN)))
tables = ab.parse_sync_tables(paths)
als = ab.alignments(tables, min_points=MIN_POINTS)
pe = []
for i, al in enumerate(als):
    d = ab.point_errors(al, s2=S2, min_fit=MIN_FIT)
    b = ab.blc_at_points(os.path.join(LOG_DIR, al['log']), list(al['points']['point_time']), al['table_time'])
    pe.append(d.assign(session=i, name=al['name'], owner=al['log'].replace('alpaca.', '').split('_')[0],
                       date=al['table_time'][:10], mac=al['mac'], blc=b['blc'].values, blc_takeups=b['takeups'].values))
pe = pd.concat(pe, ignore_index=True) if pe else pd.DataFrame()
print(f"{len(paths)} logs, {tables['log'].nunique() if len(tables) else 0} with sync tables, {len(als)} alignments of "
      f"{MIN_POINTS}+ points, {len(pe)} points; BLC active in {pe.loc[pe['blc'] != 'off', 'session'].nunique()} of them")

## 1. Which sessions' alignments are in the logs?

One row per alignment: a log's largest sync table before its point count drops (a new alignment) or its last,
without points already counted — normally one per session. *MAC* is as logged with that table (whether the driver
used it then; the replay scores both models either way). *BLC* counts the points by how Backlash Compensation ended
the goto before them ('off' = no BLC in that log). The spans show how much of the sky and of each motor the points
cover: MAC needs M2 and M3 spread to separate its offsets from the rotation.

In [ ]:
summary = pe.groupby('session').agg(name=('name', 'first'), owner=('owner', 'first'), date=('date', 'first'),
                                    mac=('mac', 'first'), n=('k', 'size'),
                                    blc=('blc', lambda b: ', '.join(f'{k} {v}' for k, v in b.value_counts().items())),
                                    az_span=('az', lambda a: np.ptp(np.unwrap(np.radians(a.astype(float))) * 180 / np.pi)),
                                    alt_min=('alt', 'min'), alt_max=('alt', 'max'),
                                    theta2_span=('theta2', np.ptp), theta3_span=('theta3', np.ptp),
                                    quest_rms=('loo_quest', rms), model_rms=('loo_model', rms))
summary['gain_pct'] = (1 - summary['model_rms'] / summary['quest_rms']) * 100
display(summary.round(1))

## 2. Once complete, how well does each point?

Each point held out in turn and predicted from all the others (leave-one-out): the alignment's accuracy for a goto
within the area it covers, once it is complete. RMS per session, in date order.

In [ ]:
s = summary.sort_values('date')
fig = go.Figure()
for side in ('quest', 'model'):
    fig.add_trace(go.Bar(x=s['name'], y=s[f'{side}_rms'], name=NAMES[side], marker_color=COLOURS[side]))
fig.update_layout(barmode='group', height=480, width=1450, template=TEMPLATE, yaxis_type='log',
                  yaxis_title="held-out goto error, RMS (arcmin, log scale)", title='Held-out accuracy per session')
fig.show()
better = int((s['model_rms'] < s['quest_rms']).sum())
print(f"median RMS: QUEST {s['quest_rms'].median():.1f}'  model {s['model_rms'].median():.1f}'  "
      f"({len(s)} sessions, the model better on {better})")
worse = s[s['model_rms'] >= s['quest_rms']]
if len(worse):
    print('MAC no better on:')
    display(worse[['name', 'n', 'theta2_span', 'theta3_span', 'quest_rms', 'model_rms']].round(1))

## 3. As the alignment grows, how well does the next goto land?

Each point predicted only from the points synced before it (from the 4th on): the goto to a new target during the
session, before its sync is added. The median across sessions (line) and the quartiles (band) against the number of
points synced so far, for the point counts that at least `MIN_SESSIONS` sessions reach (few sessions have many
points: beyond that a "median" is one or two gotos from one night). The numbers above the line are the sessions
counted. MAC fits three more terms than QUEST, so with few points it can do worse; this shows when it starts to pay,
and when more points stop helping.

In [ ]:
nx = pe.dropna(subset=['next_model'])
enough = nx.groupby('k')['session'].nunique()
enough = enough[enough >= MIN_SESSIONS].index
g = nx[nx['k'].isin(enough)].groupby('k')
fig = go.Figure()
for side in ('quest', 'model'):
    q = g[f'next_{side}'].quantile([0.25, 0.5, 0.75]).unstack()
    fig.add_trace(go.Scatter(x=list(q.index) + list(q.index[::-1]), y=list(q[0.75]) + list(q[0.25][::-1]), fill='toself',
                             line=dict(width=0), fillcolor=COLOURS[side], opacity=0.2, showlegend=False, hoverinfo='skip'))
    fig.add_trace(go.Scatter(x=q.index, y=q[0.5], mode='lines+markers', name=NAMES[side], line=dict(color=COLOURS[side]),
                             text=g['session'].nunique().astype(str) if side == 'quest' else None,
                             textposition='top center', textfont=dict(color='gray')))
    if side == 'quest':
        fig.data[-1].mode = 'lines+markers+text'
fig.update_layout(height=450, width=1000, template=TEMPLATE, yaxis_type='log',
                  xaxis_title='points synced before this goto', yaxis_title="goto error (arcmin, log scale)",
                  title='Next-goto error as the alignment grows (median, quartiles)')
fig.show()
t = g.agg(sessions=('session', 'nunique'), quest_median=('next_quest', 'median'), model_median=('next_model', 'median'),
          model_better=('next_model', lambda m: float((m < nx.loc[m.index, 'next_quest']).mean() * 100)),
          offsets_fitted=('next_offsets_fitted', lambda f: float(np.mean(f.astype(bool)) * 100)))
t.columns = ['sessions', "QUEST median '", "model median '", 'model better %', 'model fitted d2/d3 %']
display(t.round(1))
fewer = nx[~nx['k'].isin(enough)]
if len(fewer):
    print(f"left out: {fewer['k'].nunique()} point counts reached by fewer than {MIN_SESSIONS} sessions "
          f"({', '.join(sorted(fewer['name'].unique()))})")

## 4. Where does MAC help or hurt?

Each point's held-out error with QUEST against the model (points below the diagonal: MAC helped), then the change
(QUEST minus model, positive = MAC better) against where the mount was pointing and its motor angles. A trend points
at something neither model includes (e.g. a pose-dependent flexure).

In [ ]:
lim = [max(0.1, np.nanmin(pe[['loo_quest', 'loo_model']].to_numpy())), np.nanmax(pe[['loo_quest', 'loo_model']].to_numpy())]
fig = go.Figure(go.Scatter(x=pe['loo_quest'], y=pe['loo_model'], mode='markers',
                           marker=dict(size=5, color=pe['session'], colorscale='Turbo'), text=pe['name'],
                           hovertemplate="%{text}<br>QUEST %{x:.1f}' model %{y:.1f}'<extra></extra>"))
fig.add_trace(go.Scatter(x=lim, y=lim, mode='lines', line=dict(color='gray', dash='dot'), showlegend=False))
fig.update_layout(height=520, width=560, template=TEMPLATE, xaxis_type='log', yaxis_type='log', showlegend=False,
                  xaxis_title="QUEST held-out error (')", yaxis_title="model held-out error (')", title='Per point')
fig.show()

pe['mac_gain'] = pe['loo_quest'] - pe['loo_model']
XS = [('alt', 'Alt (deg)'), ('roll', 'Roll (deg)'), ('theta2', 'M2 angle θ2 (deg)'), ('theta3', 'M3 angle θ3 (deg)')]
fig = make_subplots(rows=2, cols=4, vertical_spacing=0.12,
                    subplot_titles=[f'model error vs {lab}' for _, lab in XS] + [f'MAC gain vs {lab}' for _, lab in XS])
for c, (col, lab) in enumerate(XS):
    fig.add_trace(go.Scatter(x=pe[col], y=pe['loo_model'], mode='markers', marker=dict(size=4, color=COLOURS['model']),
                             showlegend=False), row=1, col=c + 1)
    fig.add_trace(go.Scatter(x=pe[col], y=pe['mac_gain'], mode='markers', marker=dict(size=4, color='lightgreen'),
                             showlegend=False), row=2, col=c + 1)
    fig.update_xaxes(title_text=lab, row=2, col=c + 1)
fig.update_yaxes(type='log', row=1)
fig.update_yaxes(title_text="model error (')", row=1, col=1)
fig.update_yaxes(title_text="QUEST - model (')", row=2, col=1)
fig.update_layout(height=650, width=1450, template=TEMPLATE)
fig.show()

## 5. One session in detail

**Left:** where the session's sync points were on the sky as seen from the mount (azimuth around, altitude inwards,
the zenith at the centre), numbered in sync order and coloured by the model's error (arcmin, colour bar).
**Right:** each point's error as a dart (arcmin): where the star was relative to where the alignment pointed —
horizontal = ΔAz × cos(Alt), vertical = ΔAlt — for QUEST and the model, with rings at 5', 10' and 30'. The two
panels have their own scales: the sky map is in degrees, the dartboard in arcminutes.

`KIND` picks the held-out errors ('loo', section 2) or the next-goto errors ('next', section 3). Step through the
sessions with the slider; `SESSION` picks the first shown.

In [ ]:
KIND = 'loo'                    # 'loo' (held out from all the others) or 'next' (from the earlier points only)
COLOUR_MAX = 15                 # arcmin: the top of the sky map's colour scale
RINGS = [5, 10, 30]             # arcmin

def session_traces(d, kind=KIND):
    """The sky map's points (polar) and the dartboard's darts (xy) for one session."""
    out = [go.Scatterpolar(r=90 - d['alt'].astype(float), theta=d['az'].astype(float), mode='markers+text',
                           text=d['k'].astype(str), textposition='top center', showlegend=False,
                           marker=dict(size=11, color=d[f'{kind}_model'], cmin=0, cmax=COLOUR_MAX, colorscale='Turbo',
                                       colorbar=dict(title="model<br>error '", x=0.44, len=0.8)),
                           customdata=np.c_[d[f'{kind}_model'], d[f'{kind}_quest'], d['blc']],
                           hovertemplate="point %{text}: model %{customdata[0]:.1f}' QUEST %{customdata[1]:.1f}' "
                                         "BLC %{customdata[2]}<extra></extra>", subplot='polar')]
    for side in ('quest', 'model'):
        out.append(go.Scatter(x=d[f'{kind}_{side}_h'], y=d[f'{kind}_{side}_v'], mode='markers+text',
                              text=d['k'].astype(str), textposition='top center',
                              marker=dict(size=9, color=COLOURS[side]),
                              name=f"{NAMES[side]}: RMS {rms(d[f'{kind}_{side}']):.1f}'", xaxis='x', yaxis='y'))
    return out

def dart_range(d, kind=KIND):
    e = d[[f'{kind}_quest_h', f'{kind}_quest_v', f'{kind}_model_h', f'{kind}_model_v']].abs().to_numpy(float)
    return max(RINGS[0], np.nanmax(e) if np.isfinite(e).any() else RINGS[0]) * 1.15

if len(als):
    active = SESSION % len(als)
    fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'polar'}, {'type': 'xy'}]], column_widths=[0.5, 0.5],
                        subplot_titles=['sync points on the sky', "errors (arcmin)"])
    counts, ranges = [], []
    for i in range(len(als)):
        d = pe[pe['session'] == i]
        tr = session_traces(d)
        counts.append(len(tr))
        ranges.append(dart_range(d))
        for t in tr:
            fig.add_trace(t)
    rings = len(fig.data)
    a = np.linspace(0, 2 * np.pi, 120)
    for r in RINGS:
        fig.add_trace(go.Scatter(x=r * np.cos(a), y=r * np.sin(a), mode='lines', line=dict(color='gray', width=0.7, dash='dot'),
                                 hoverinfo='skip', showlegend=False, xaxis='x', yaxis='y'))
    starts = np.cumsum([0] + counts[:-1])
    steps = []
    for i, (s0, n) in enumerate(zip(starts, counts)):
        vis = [False] * rings + [True] * len(RINGS)
        vis[s0:s0 + n] = [True] * n
        steps.append(dict(method='update', label=str(i),
                          args=[{'visible': vis}, {'title.text': f"{i}: {als[i]['name']} ({KIND})",
                                                   'xaxis.range': [-ranges[i], ranges[i]], 'yaxis.range': [-ranges[i], ranges[i]]}]))
    for j, t in enumerate(fig.data[:rings]):
        t.visible = starts[active] <= j < starts[active] + counts[active]
    fig.update_xaxes(range=[-ranges[active], ranges[active]], zeroline=True, title_text='horizontal: ΔAz·cos(Alt) (arcmin)')
    fig.update_yaxes(range=[-ranges[active], ranges[active]], zeroline=True, scaleanchor='x', title_text='vertical: ΔAlt (arcmin)')
    fig.update_layout(sliders=[dict(active=active, steps=steps, currentvalue=dict(prefix='session: '), pad=dict(t=50))],
                      polar=dict(radialaxis=dict(range=[0, 90], tickvals=[0, 30, 60, 90], ticktext=['90°', '60°', '30°', '0°']),
                                 angularaxis=dict(rotation=90, direction='clockwise')),
                      height=700, width=1400, template=TEMPLATE, title=f"{active}: {als[active]['name']} ({KIND})",
                      legend=dict(x=0.55, y=1.0))
    fig.show()
    display(pe[pe['session'] == active][['k', 'point_time', 'az', 'alt', 'roll', 'theta2', 'theta3', 'blc', 'loo_quest',
                                         'loo_model', 'next_quest', 'next_model']].round(1))

## 6. What M2 scale and offsets do the alignments want?

The model's held-out RMS for a range of fixed M2 scales (median over the sessions), then each session fitted on all
its points with the scale free: the s2 and the offsets d2, d3 it settles on, in date order. The offsets move with the
setup (how the camera sits on the head, and possibly each power-up's motor zero, System Design §5.6); the M2 scale
should be a property of the mount.

In [ ]:
SCALES = [0.0, 0.005, 0.01, 0.015, 0.02, 0.03]
sweep = []
for s2 in SCALES:
    r = [rms(ab.loo_errors_arcmin(al, s2=s2)['model']) for al in als]
    sweep.append({'M2 scale %': s2 * 100, "median model RMS '": np.median(r), '_r': r})
best = np.argmin(np.array([w['_r'] for w in sweep]), axis=0)
for i, w in enumerate(sweep):
    w['best for sessions'] = int(np.sum(best == i))
display(pd.DataFrame(sweep).drop(columns='_r').set_index('M2 scale %').round(2))

fits = []
for i, al in enumerate(als):
    f = fit_alignment(al['theta'], al['obs'], s2=S2, fit_s2=True)
    fits.append({'session': i, 'name': al['name'], 'date': al['table_time'][:10], 'n': len(al['obs']),
                 'offsets fitted': f.offsets_fitted, 's2 fitted': f.s2_fitted, 'M2 scale %': f.s2 * 100,
                 "d2 '": f.d2 * 60, "d3 '": f.d3 * 60, "rms '": f.rms_arcmin})
fits = pd.DataFrame(fits).sort_values('date')
display(fits.set_index('session').round(2))
ok = fits[fits['offsets fitted']]
fig = make_subplots(rows=1, cols=3, subplot_titles=["d2 (')", "d3 (')", 'M2 scale (%)'])
for c, col in enumerate(["d2 '", "d3 '", 'M2 scale %']):
    d = ok if col != 'M2 scale %' else fits[fits['s2 fitted']]
    fig.add_trace(go.Scatter(x=d['date'], y=d[col], mode='markers', text=d['name'], marker=dict(size=8),
                             showlegend=False), row=1, col=c + 1)
fig.update_layout(height=420, width=1450, template=TEMPLATE, title='Fitted on all points, scale free')
fig.show()

## 7. What made a session point well: the offsets, MAC or BLC?

Three things can make a session's alignment good, and they are easy to confuse:

* **Small M2/M3 zero offsets (d2, d3).** QUEST is a rotation only: it cannot fit the offsets, so its error grows
  with their size. MAC fits them, so its error shouldn't. **Left:** each session's held-out RMS against the size of
  its offsets (√(d2² + d3²), fitted on all its points with the M2 scale at `S2`).
* **MAC** — the gap between the two colours at the same offset size.
* **Backlash Compensation.** BLC makes each goto finish with every motor engaged the way it tracks, so the raw motor
  angles in the sync table describe the output shafts. Without it a sync can be off by up to a motor's slack
  (`backlash_deg`: ~2' M1, ~7' M2, ~5' M3), different from point to point, which no model can fit. So BLC should lower
  *both* models' error by up to a few arcminutes — it can't remove tens of arcminutes of offsets. Sessions with BLC
  are ringed in white. **Right:** each session's MAC held-out RMS, sessions without BLC against sessions with it
  (one dot per session; with MAC the offsets no longer matter, so this is where BLC's few arcminutes would show).
  The table under it splits the BLC sessions' points by how BLC ended the goto before each.

BLC's main effect is on Slew & Center's small correction gotos rather than the alignment; `analyse_scc.ipynb` measures
that.

In [ ]:
fs = fits.set_index('session')
sess = summary.join(fs[["d2 '", "d3 '", 'offsets fitted']])
sess['offsets'] = np.hypot(sess["d2 '"], sess["d3 '"])
sess['BLC'] = pe.groupby('session')['blc'].agg(lambda b: bool((b != 'off').any()))
sx = sess[sess['offsets fitted']]

fig = make_subplots(rows=1, cols=2, column_widths=[0.6, 0.4],
                    subplot_titles=['held-out RMS against the size of the M2/M3 offsets', 'MAC held-out RMS: without and with BLC'])
for side in ('quest', 'model'):
    fig.add_trace(go.Scatter(x=sx['offsets'], y=sx[f'{side}_rms'], mode='markers', name=NAMES[side], text=sx['name'],
                             marker=dict(size=9, color=COLOURS[side], line=dict(width=np.where(sx['BLC'], 3, 0), color='white')),
                             hovertemplate="%{text}<br>offsets %{x:.0f}' RMS %{y:.1f}'<extra></extra>"), row=1, col=1)
blc_pts = pe[pe['blc'] != 'off']
for on, d in sess.groupby('BLC'):
    label = f"{'with' if on else 'without'} BLC<br>({len(d)} sessions)"
    fig.add_trace(go.Box(y=d['model_rms'], name=label, text=d['name'], boxpoints='all', jitter=0.4, pointpos=0,
                         marker=dict(color='lightgreen' if on else 'gray', size=8), line=dict(color='lightgreen' if on else 'gray'),
                         hovertemplate="%{text}<br>MAC RMS %{y:.1f}'<extra></extra>", showlegend=False), row=1, col=2)
fig.update_xaxes(title_text="√(d2² + d3²) (arcmin)", row=1, col=1)
fig.update_yaxes(type='log', title_text="held-out RMS (arcmin)", row=1, col=1)
fig.update_yaxes(type='log', title_text="MAC held-out RMS (arcmin)", row=1, col=2)
fig.update_layout(height=520, width=1450, template=TEMPLATE, title='Sessions with BLC are ringed white')
fig.show()

for side in ('quest', 'model'):
    r = np.corrcoef(sx['offsets'], sx[f'{side}_rms'])[0, 1]
    print(f"{NAMES[side]}: correlation of the session's RMS with its offsets {r:+.2f}")
print(f"sessions with BLC: {int(sess['BLC'].sum())} of {len(sess)}")
display(sess[['name', 'n', 'mac', 'BLC', "d2 '", "d3 '", 'offsets', 'quest_rms', 'model_rms']]
        .sort_values('offsets').round(1).head(10))
print('MAC held-out RMS, median: ' + ', '.join(f"{'with' if on else 'without'} BLC {d['model_rms'].median():.1f}' "
                                               f"({len(d)} sessions)" for on, d in sess.groupby('BLC')))
if len(blc_pts):
    print('BLC sessions, points by how BLC ended the goto before them (held-out error, arcmin):')
    display(blc_pts.groupby('blc')[['loo_quest', 'loo_model']].agg(['count', 'median']).round(1))